In [1]:
# ============================================================
# 终极策略终端 (SaaS安全版)：自定义科技股标的池 + 全屏对齐 + 极速渲染
# ============================================================
import math
import time
import datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import longport.openapi as lp 

# 🌟 全局上下文变量 (登录前为空)
ctx = None

# --- 1. 辅助逻辑 ---
def interp_flip(df_net):
    if df_net is None or df_net.empty: return None
    df_net = df_net.sort_values("Strike").reset_index(drop=True)
    xs, ys = df_net["Strike"].values, df_net["NetGEX"].values
    for i in range(len(df_net) - 1):
        if ys[i] * ys[i+1] < 0:
            return float(xs[i] + (-ys[i]) * (xs[i+1] - xs[i]) / (ys[i+1] - ys[i]))
    return None

def create_pagination_widget(df, rows_per_page=15):
    if df is None or df.empty: return widgets.Label("⚠️ 无数据")
    df_raw = df.copy()
    page_input = widgets.BoundedIntText(value=1, min=1, max=100, description='页码:', layout={'width': '90px'})
    prev_btn = widgets.Button(description="上一页", layout={'width': '75px'})
    next_btn = widgets.Button(description="下一页", layout={'width': '75px'})
    sort_dropdown = widgets.Dropdown(options=['原始顺序']+list(df.columns), value='原始顺序', description='排序:', layout={'width': '180px'})
    sort_order = widgets.Dropdown(options=[('降序 ⬇️', False), ('升序 ⬆️', True)], value=False, description='方式:', layout={'width': '120px'})
    output = widgets.Output(layout={'width': '100%'})
    def show_page(page):
        view = df_raw if sort_dropdown.value == '原始顺序' else df_raw.sort_values(by=sort_dropdown.value, ascending=sort_order.value)
        page_input.max = max(1, (len(view)-1)//rows_per_page + 1)
        with output: clear_output(wait=True); display(view.iloc[(page-1)*rows_per_page : page*rows_per_page])
    def on_ui_change(change): show_page(page_input.value)
    prev_btn.on_click(lambda _: setattr(page_input, 'value', max(1, page_input.value-1)))
    next_btn.on_click(lambda _: setattr(page_input, 'value', min(page_input.max, page_input.value+1)))
    for w in [page_input, sort_dropdown, sort_order]: w.observe(on_ui_change, names='value')
    show_page(1); return widgets.VBox([widgets.HBox([prev_btn, next_btn, page_input, sort_dropdown, sort_order]), output], layout={'width': '100%'})

# --- 2. 核心组件 (分离 HTML 与 Figure 避免报错) ---
def render_k_line(symbol):
    try:
        kline_symbol = '.INX' if symbol == 'SPX.US' else symbol
        klines = ctx.candlesticks(kline_symbol, lp.Period.Day, 1000, lp.AdjustType.ForwardAdjust)
        df = pd.DataFrame([{"Date": k.timestamp, "O": float(k.open), "H": float(k.high), "L": float(k.low), "C": float(k.close), "V": int(k.volume or 0)} for k in klines])
        df['DateStr'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
        df = df.sort_values('DateStr').reset_index(drop=True)
        
        last_row = df.iloc[-1]
        last_c = last_row['C']
        prev_p = df['C'].iloc[-2] if len(df) > 1 else last_row['O']
        diff = last_c - prev_p
        pct = diff / prev_p * 100 if prev_p != 0 else 0
        color_theme = "#00C805" if diff >= 0 else "#FF3B30"
        vol = last_row['V']
        vol_str = f"{vol/1e8:.2f}亿" if vol >= 1e8 else (f"{vol/1e4:.2f}万" if vol >= 1e4 else f"{vol}") if vol > 0 else "--"

        html_str = f"""
        <div style="font-family: Arial, sans-serif; padding-bottom: 8px; border-bottom: 2px solid #e0e0e0; margin-bottom: 5px; width: 100%;">
            <span style="font-size: 20px; font-weight: 900; color: #1a237e;">{symbol}</span>
            <span style="margin-left: 20px; font-size: 14px; font-weight: bold; background: #f0f0f0; padding: 2px 8px; border-radius: 4px;">{last_row['DateStr']}</span>
            <span style="margin-left: 15px; font-size: 14px;">开 <span style="color:{color_theme}; font-weight:bold;">{last_row['O']:.2f}</span></span>
            <span style="margin-left: 12px; font-size: 14px;">高 <span style="color:{color_theme}; font-weight:bold;">{last_row['H']:.2f}</span></span>
            <span style="margin-left: 12px; font-size: 14px;">低 <span style="color:{color_theme}; font-weight:bold;">{last_row['L']:.2f}</span></span>
            <span style="margin-left: 12px; font-size: 14px;">收 <span style="color:{color_theme}; font-weight:bold;">{last_row['C']:.2f}</span></span>
            <span style="margin-left: 12px; font-size: 14px;">幅 <span style="color:{color_theme}; font-weight:bold;">{diff:+.2f} ({pct:+.2f}%)</span></span>
            <span style="margin-left: 12px; font-size: 14px;">量 <span style="color:#333; font-weight:bold;">{vol_str}</span></span>
        </div>
        """

        fig = go.Figure(make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.8, 0.2]))
        
        candlestick = go.Candlestick(x=df['DateStr'], open=df['O'], high=df['H'], low=df['L'], close=df['C'], name=symbol,
                                     increasing_line_color='#00C805', increasing_fillcolor='rgba(0,0,0,0)', 
                                     decreasing_line_color='#FF3B30', decreasing_fillcolor='#FF3B30', line_width=1.0)
        fig.add_trace(candlestick, row=1, col=1)
        
        v_clrs = ['#1B5E20' if c>=o else '#B71C1C' for o,c in zip(df['O'],df['C'])]
        vol_bar = go.Bar(x=df['DateStr'], y=df['V'], name='Volume', marker_color=v_clrs, opacity=1.0, marker=dict(line=dict(width=0)))
        fig.add_trace(vol_bar, row=2, col=1)
        
        fig.add_hline(y=last_c, line=dict(color='gray', dash='dash', width=1), row=1, col=1)
        fig.add_annotation(xref="paper", yref="y", x=1.0, y=last_c, text=f"<b>{last_c:.2f}</b>", showarrow=False, xanchor="left", bgcolor="black", font=dict(color="white", size=12), borderpad=4)
        
        fig.update_xaxes(type='category', showspikes=True, spikecolor="rgba(128,128,128,0.4)", spikethickness=1, spikedash="dash", spikemode="across")
        fig.update_yaxes(showspikes=True, spikecolor="rgba(128,128,128,0.4)", spikethickness=1, spikedash="dash", spikemode="across")
        
        fig.update_layout(
            autosize=True,
            bargap=0.25,
            height=480, template='plotly_white', xaxis_rangeslider_visible=False, dragmode='pan',
            hovermode='x unified', 
            hoverlabel=dict(bgcolor="rgba(255, 255, 255, 0.45)", font_size=12, font_color="black"),
            margin=dict(t=10, b=10, l=10, r=60) 
        )
        fig.update_yaxes(autorange=True, fixedrange=False, row=1, col=1); fig.update_yaxes(autorange=True, fixedrange=False, row=2, col=1)

        return html_str, fig
    except Exception as e: 
        return f"<b style='color:red'>K线加载失败: {e}</b>", None

def render_sentiment():
    try:
        temp = ctx.market_temperature(lp.Market.US)
        val = float(getattr(temp, 'temperature', getattr(temp, 'cur', 50)))
        name = getattr(temp, 'cur_name', None)
        if not name or str(name).strip() in ['', '未知', 'None']:
            if val < 25: name = '极度恐慌'
            elif val < 45: name = '恐慌'
            elif val < 55: name = '中性'
            elif val < 75: name = '贪婪'
            else: name = '极度贪婪'
            
        fig = go.Figure(go.Indicator(mode="gauge+number", value=val, 
                                     title={'text': f"市场温度: <b>{name}</b>"}, 
                                     gauge={'axis':{'range':[0,100]}, 'bar':{'color':"#2c3e50"},
                                            'steps':[{'range':[0,30],'color':"#FF3B30"},{'range':[30,70],'color':"#FAD02E"},{'range':[70,100],'color':"#00C805"}]}))
        fig.update_layout(autosize=True, height=180, margin=dict(t=40,b=10)); return fig
    except Exception as e: 
        return widgets.Label(f"情绪加载中: {e}")

# --- 3. 终端 UI 定义 (全部强制 100% 宽度，更新标的池) ---
# 🟢 删除了 SPX，加入了 QQQ, AMZN, META, MU
SYMBOL_LIST = ['SPY.US', 'QQQ.US', 'AMZN.US', 'META.US', 'MU.US', 'TSLA.US', 'NVDA.US', 'AAPL.US', 'GOOG.US']

symbol_sel = widgets.Dropdown(options=SYMBOL_LIST, value='SPY.US', description='Ticker:', layout={'width':'240px'})
date_sel = widgets.SelectMultiple(description="Expiry:", layout={'width':'460px','height':'120px'})
win_s = widgets.IntSlider(value=40, min=10, max=150, description="Window")
rate_s = widgets.FloatSlider(value=0.037, min=0, max=0.1, step=0.0001, description="Rate", readout_format=".2%")
div_s = widgets.FloatSlider(value=0.012, min=0, max=0.05, step=0.0001, description="Div", readout_format=".2%")
run_btn = widgets.Button(description="SYNC DASHBOARD", button_style="success", icon="bolt", layout={'width':'180px'})

k_out = widgets.Output(layout={'width': '100%'})
t_out = widgets.Output(layout={'width': '100%'})
g_out = widgets.Output(layout={'width': '100%'})

terminal_ui = widgets.VBox([
    widgets.HTML("<h1 style='text-align:center;'>ULTIMATE STRATEGY TERMINAL</h1>"),
    widgets.HBox([widgets.HTML("<h2>选择标的: </h2>"), symbol_sel]),
    k_out,
    widgets.HTML("<hr><h3 style='padding-left:10px;'>📊 市场情绪看板</h3>"),
    t_out,
    widgets.HTML("<hr><h3 style='padding-left:10px;'>🔍 期权链深度分析</h3>"),
    date_sel,
    widgets.HBox([win_s, rate_s, div_s, run_btn], layout={'align_items':'center', 'margin':'10px 0'}),
    g_out
], layout=widgets.Layout(width='100%'))
terminal_ui.layout.display = 'none'

# --- 4. 终端业务逻辑 ---
def on_symbol_change(change):
    ticker = symbol_sel.value
    try:
        today = dt.date.today()
        all_dates = ctx.option_chain_expiry_date_list(ticker)
        valid_dates = [d for d in all_dates if d >= today]
        date_sel.options = [(str(d), d) for d in valid_dates[:20]]
        if valid_dates: date_sel.value = (valid_dates[0],)
    except: pass
    
    with k_out: 
        clear_output(wait=True)
        html_str, fig = render_k_line(ticker)
        display(widgets.HTML(value=html_str, layout={'width': '100%'}))
        if fig: display(fig)
        
    with t_out:
        clear_output(wait=True); res_t = render_sentiment()
        if hasattr(res_t, 'show'): res_t.show()
        else: display(res_t)
    with g_out: clear_output(wait=True); print("已切换标的。请选择到期日并点击 [SYNC DASHBOARD] 分析期权...")

def on_sync_click(_):
    ticker = symbol_sel.value
    with g_out:
        clear_output(wait=True)
        try:
            print(f"🚀 正在抓取 {ticker} 期权数据...")
            quote_symbol = '.INX' if ticker == 'SPX.US' else ticker
            spot = float(ctx.quote([quote_symbol])[0].last_done)
            all_rows, sym_map = [], {}
            for exp in list(date_sel.value):
                chain = ctx.option_chain_info_by_date(ticker, exp)
                for item in (chain or []):
                    if abs(float(item.price) - spot) <= win_s.value:
                        sym_map[item.call_symbol] = {"K": float(item.price), "T": "C", "Exp": exp}
                        sym_map[item.put_symbol] = {"K": float(item.price), "T": "P", "Exp": exp}
            
            if not sym_map: print("⚠️ 未抓取到期权。请尝试调大 Window 范围。"); return

            symbols = list(sym_map.keys())
            indices_req = [lp.CalcIndex.Gamma, lp.CalcIndex.Delta, lp.CalcIndex.Theta, lp.CalcIndex.Vega, lp.CalcIndex.ImpliedVolatility]
            for i in range(0, len(symbols), 100):
                batch = symbols[i:i+100]; calc_res = ctx.calc_indexes(batch, indices_req)
                quotes = ctx.option_quote(batch)
                oi_dict = {q.symbol: int(q.open_interest or 0) for q in quotes}
                vol_dict = {q.symbol: int(q.volume or 0) for q in quotes}
                
                for res in calc_res:
                    m = sym_map[res.symbol]
                    oi = oi_dict.get(res.symbol, 0); vol = vol_dict.get(res.symbol, 0); g = float(res.gamma or 0.0)
                    iv = float(res.implied_volatility or 0.0); iv = iv/100.0 if iv > 1.5 else iv
                    all_rows.append({"Strike": m["K"], "Type": m["T"], "Expiry": m["Exp"], "Delta": float(res.delta or 0.0), 
                                     "Gamma": g, "Theta": float(res.theta or 0.0), "Vega": float(res.vega or 0.0),
                                     "IV": iv, "OI": oi, "Volume": vol, "RawGEX": (g * oi * 100 * (spot**2) * 0.01) / 1e6})
            
            df = pd.DataFrame(all_rows)
            
            # --- 看板数据 ---
            total_vol = df['Volume'].sum(); call_df = df[df['Type'] == 'C']; put_df = df[df['Type'] == 'P']
            c_vol = call_df['Volume'].sum(); p_vol = put_df['Volume'].sum()
            pc_vol_ratio = p_vol / c_vol if c_vol > 0 else 0
            total_oi = df['OI'].sum(); c_oi = call_df['OI'].sum(); p_oi = put_df['OI'].sum()
            pc_oi_ratio = p_oi / c_oi if c_oi > 0 else 0
            atm_strike = df.iloc[(df['Strike'] - spot).abs().argsort()[:1]]['Strike'].values[0]
            atm_iv = df[df['Strike'] == atm_strike]['IV'].mean() * 100
            
            stats_html = f"""
            <div style="display:flex; justify-content:space-around; background-color:#f9fbff; padding:15px; border-radius:8px; border:1px solid #d0d7e5; margin-bottom:15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
                <div style="text-align:center;"><span style="color:#666; font-size:13px;">总成交量</span><br><b style="font-size:18px; color:#1a237e;">{total_vol/10000:.2f}万</b></div>
                <div style="text-align:center;"><span style="color:#666; font-size:13px;">P/C 成交比</span><br><b style="font-size:18px; color:#1a237e;">{pc_vol_ratio:.2f}</b></div>
                <div style="text-align:center;"><span style="color:#666; font-size:13px;">总持仓量 (OI)</span><br><b style="font-size:18px; color:#1a237e;">{total_oi/10000:.2f}万</b></div>
                <div style="text-align:center;"><span style="color:#666; font-size:13px;">P/C 持仓比</span><br><b style="font-size:18px; color:#1a237e;">{pc_oi_ratio:.2f}</b></div>
                <div style="text-align:center;"><span style="color:#666; font-size:13px;">ATM IV (平值)</span><br><b style="font-size:18px; color:#1a237e;">{atm_iv:.2f}%</b></div>
            </div>
            """
            display(widgets.HTML(stats_html, layout={'width': '100%'}))

            # --- IV Curve 图表 ---
            fig_iv = go.Figure()
            c_iv_curve = call_df[call_df['IV'] < 3.0].groupby('Strike')['IV'].mean().reset_index()
            p_iv_curve = put_df[put_df['IV'] < 3.0].groupby('Strike')['IV'].mean().reset_index()
            fig_iv.add_trace(go.Scatter(x=c_iv_curve['Strike'], y=c_iv_curve['IV']*100, mode='lines+markers', name='Call IV', line=dict(color='#00C805', width=2), marker=dict(size=5)))
            fig_iv.add_trace(go.Scatter(x=p_iv_curve['Strike'], y=p_iv_curve['IV']*100, mode='lines+markers', name='Put IV', line=dict(color='#FF3B30', width=2), marker=dict(size=5)))
            fig_iv.add_vline(x=spot, line=dict(color='gray', dash='dash'), annotation_text=f"Spot: {spot:.2f}", annotation_position="top right")
            fig_iv.update_layout(autosize=True, title=f"<b>{ticker} IV Curve (波动率微笑)</b>", height=350, template="plotly_white", xaxis_title="Strike (行权价)", yaxis_title="Implied Volatility (%)", hovermode="x unified", margin=dict(t=40, b=10, l=10, r=10))
            display(fig_iv)

            # --- GEX 图表 ---
            agg = df.groupby("Strike").apply(lambda x: pd.Series({
                'CallGEX': x[x['Type']=='C']['RawGEX'].sum(), 'PutGEX': x[x['Type']=='P']['RawGEX'].sum(),
                'CallGamma': x[x['Type']=='C']['Gamma'].sum(), 'PutGamma': x[x['Type']=='P']['Gamma'].sum(),
                'NetGEX': x[x['Type']=='C']['RawGEX'].sum() - x[x['Type']=='P']['RawGEX'].sum()
            }), include_groups=False).reset_index()

            flip = interp_flip(agg); cw = agg.loc[agg['CallGEX'].idxmax(), 'Strike']; pw = agg.loc[agg['PutGEX'].idxmax(), 'Strike']
            max_abs_gex = max(agg['NetGEX'].abs().max(), 1.0)
            fig_gex = go.Figure(go.Bar(y=agg["Strike"], x=agg["NetGEX"], orientation="h", marker_color=["#00C805" if x >= 0 else "#FF3B30" for x in agg["NetGEX"]]))
            markers = [(spot,"Spot","blue"),(cw,"Call Wall","#00C805"),(pw,"Put Wall","#FF3B30"),(flip,"Flip","purple")]
            for v,n,c in markers:
                if v: fig_gex.add_hline(y=v, line=dict(color=c, width=1.5, dash="dash" if n!="Spot" else "solid"), annotation_text=f" {n}: {v:.2f}")
            fig_gex.update_layout(autosize=True, title=f"<b>{ticker} GEX PROFILE (Mirror Axis)</b>", height=600, template="plotly_white", xaxis=dict(range=[-max_abs_gex*1.1, max_abs_gex*1.1], zeroline=True))
            display(fig_gex)
            
            print("\n===== Strike Aggregated GEX / Gamma ====="); display(create_pagination_widget(agg.round(4)))
            print("\n===== Option Details (希腊字母) ====="); display(create_pagination_widget(df.round(6)))
        except Exception as e: print(f"❌ 同步失败: {e}")

symbol_sel.observe(on_symbol_change, names='value')
run_btn.on_click(on_sync_click)

# --- 5. 登录界面 UI 定义 ---
login_title = widgets.HTML(
    "<h1 style='text-align:center; color:#1a237e; font-family:sans-serif;'>🔒 量化策略终端接入</h1>"
    "<p style='text-align:center; color:#666; font-size:14px;'>检测到当前环境未授权，请在此输入您的 LongPort API 凭证</p>"
)
in_app_key = widgets.Password(description="App Key:", layout={'width':'350px'})
in_app_sec = widgets.Password(description="App Secret:", layout={'width':'350px'})
in_token = widgets.Password(description="Token:", layout={'width':'350px'})
btn_login = widgets.Button(description="连接服务器 (Connect)", button_style="info", layout={'width':'350px', 'margin':'15px 0 0 0'})
msg_login = widgets.HTML("", layout={'text_align':'center', 'width':'350px'})

login_ui = widgets.VBox([login_title, in_app_key, in_app_sec, in_token, btn_login, msg_login], 
                        layout=widgets.Layout(align_items='center', margin='80px 0', border='1px solid #e0e0e0', padding='30px', border_radius='10px', box_shadow='0 4px 8px rgba(0,0,0,0.1)'))

# --- 6. 登录验证逻辑 ---
def handle_login(b):
    global ctx
    btn_login.description = "正在验证..."
    btn_login.disabled = True
    msg_login.value = ""
    
    try:
        temp_config = lp.Config(app_key=in_app_key.value, app_secret=in_app_sec.value, access_token=in_token.value)
        temp_ctx = lp.QuoteContext(temp_config)
        temp_ctx.market_temperature(lp.Market.US)
        ctx = temp_ctx
        
        login_ui.layout.display = 'none'
        terminal_ui.layout.display = 'block'
        
        time.sleep(0.1) 
        on_symbol_change(None)
        
    except Exception as e:
        btn_login.description = "连接服务器 (Connect)"
        btn_login.disabled = False
        msg_login.value = f"<span style='color:red; font-size:12px;'>❌ 验证失败，请检查凭证是否正确或已过期。</span>"

btn_login.on_click(handle_login)

# --- 7. 主程序入口 ---
main_app = widgets.VBox([login_ui, terminal_ui], layout=widgets.Layout(width='100%'))
display(main_app)